# Intelligent Document Analysis Agent for Resume Screening and Information Extraction
**AI Mini-Capstone (Artificial Intelligence - 25CS301) | Topic 3.5: Real-Time AI Agents and Applications**

**AI method used:** a *Utility-Based, Knowledge-Based Intelligent Agent* (Perceive -> Reason with a knowledge base -> Act).

**How to run on Google Colab:** `Runtime -> Run all`. The notebook generates the 50,000-resume dataset by itself (fixed random seed, so results are reproducible).
If you already have `resumes_50000.csv`, upload it to the Colab file panel first and the notebook will use it instead.

**Pipeline**
1. Build the knowledge base (skills + synonyms, degrees, section headings, job profile, scoring rules)
2. Create / load the 50,000-resume dataset (synthetic, with hidden ground truth for evaluation)
3. Build the agent (Sensors -> Reasoning -> Actuators)
4. Run the agent on all 50,000 resumes
5. Evaluate: extraction accuracy, decision accuracy, confusion matrix
6. Compare with alternative approaches (keyword matching, TF-IDF similarity) and run an ablation study
7. Error analysis, ranked shortlist, and a live demo cell

In [ ]:
import re, random, time, os, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
N_RESUMES = 50_000
DATA_PATH = "resumes_50000.csv"
NOW = (2026, 11)          # reference "today" (year, month) used for jobs marked "Present"
random.seed(SEED); np.random.seed(SEED)
print("Setup complete")

## Step 1: Knowledge Base of the Agent
A *knowledge-based agent* needs explicit knowledge. Ours has five parts:
1. **Skill ontology:** each skill with its synonyms/abbreviations (e.g. `ML` = `machine learning`)
2. **Degree knowledge:** which degree strings mean Bachelor/Master and whether they are technical
3. **Section knowledge:** which headings mean Experience / Education / Skills ...
4. **Job profile:** required skills, preferred skills, minimum experience
5. **Decision rules + utility function:** how a candidate is scored and which action is taken

In [ ]:
# 1) Skill ontology: canonical skill -> aliases (first alias is always the canonical name)
SKILL_KB = {
    "python": ["python", "python3"],
    "java": ["java"],
    "c++": ["c++", "cpp"],
    "javascript": ["javascript", "js"],
    "sql": ["sql"],
    "mysql": ["mysql"],
    "mongodb": ["mongodb", "mongo db"],
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning"],
    "nlp": ["nlp", "natural language processing"],
    "computer vision": ["computer vision"],
    "tensorflow": ["tensorflow"],
    "pytorch": ["pytorch", "torch"],
    "scikit-learn": ["scikit-learn", "sklearn"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "data analysis": ["data analysis", "data analytics"],
    "statistics": ["statistics", "statistical analysis"],
    "power bi": ["power bi", "powerbi"],
    "tableau": ["tableau"],
    "excel": ["excel", "ms excel"],
    "aws": ["aws", "amazon web services"],
    "azure": ["azure", "microsoft azure"],
    "docker": ["docker"],
    "kubernetes": ["kubernetes", "k8s"],
    "git": ["git"],
    "linux": ["linux"],
    "html": ["html", "html5"],
    "css": ["css", "css3"],
    "react": ["react", "reactjs", "react.js"],
    "node.js": ["node.js", "nodejs"],
    "django": ["django"],
    "flask": ["flask"],
    "apache spark": ["apache spark", "pyspark"],
}

# 4) Job profile: AI / Machine Learning Engineer
JOB_TITLE  = "AI / Machine Learning Engineer"
REQUIRED   = ["python", "machine learning", "sql"]
PREFERRED  = ["deep learning", "nlp", "tensorflow", "pytorch", "scikit-learn", "pandas", "aws", "docker", "git"]
MIN_EXP_YEARS = 3.0
WEIGHTS = {"req": 0.35, "pref": 0.25, "exp": 0.25, "deg": 0.15}
SHORTLIST_T, REVIEW_T = 0.65, 0.45

# 2) Degree knowledge: (regex, level, is_technical, label)
DEGREE_KB = [
    (r"\bm\.?\s?tech\b|master of technology",           "Master",   True,  "M.Tech"),
    (r"\bmca\b|master of computer applications",         "Master",   True,  "MCA"),
    (r"\bm\.?\s?sc\b|master of science",                 "Master",   True,  "M.Sc"),
    (r"\bmba\b|master of business administration",       "Master",   False, "MBA"),
    (r"\bb\.?\s?tech\b|bachelor of technology|\bb\.e\.?|bachelor of engineering", "Bachelor", True, "B.Tech / B.E"),
    (r"\bbca\b|bachelor of computer applications",       "Bachelor", True,  "BCA"),
    (r"\bb\.?\s?sc\b|bachelor of science",               "Bachelor", True,  "B.Sc"),
    (r"\bb\.?\s?com\b|bachelor of commerce|\bbba\b|bachelor of business administration|\bb\.a\.?|bachelor of arts",
                                                          "Bachelor", False, "Non-technical Bachelor"),
]
DEGREE_KB = [(re.compile(rx, re.I), lvl, tech, lab) for rx, lvl, tech, lab in DEGREE_KB]
DEGREE_SCORE = {("Master", True): 1.0, ("Bachelor", True): 0.9, ("Master", False): 0.4, ("Bachelor", False): 0.3}

# 3) Section knowledge: heading text -> canonical section
HEADINGS = {
    "summary": ["summary", "profile", "career objective"],
    "skills":  ["skills", "technical skills", "core competencies", "key skills"],
    "exp":     ["experience", "work experience", "professional experience", "employment history"],
    "edu":     ["education", "academic background"],
    "proj":    ["projects", "academic projects"],
    "cert":    ["certifications"],
}
HEADING_MAP = {h: sec for sec, hs in HEADINGS.items() for h in hs}

# 5) Utility function + decision rules (shared by the agent and by the ground-truth labeller)
def utility(skills, years, deg_score):
    req_n  = sum(s in skills for s in REQUIRED)
    pref_n = sum(s in skills for s in PREFERRED)
    u = (WEIGHTS["req"]  * req_n / len(REQUIRED)
       + WEIGHTS["pref"] * min(1.0, pref_n / 5)
       + WEIGHTS["exp"]  * min(1.0, years / MIN_EXP_YEARS)
       + WEIGHTS["deg"]  * deg_score)
    return round(u, 4), req_n, pref_n

def decide(u, req_n):
    if req_n == 0:          return "REJECT"        # knock-out rule: none of the required skills
    if u >= SHORTLIST_T:    return "SHORTLIST"
    if u >= REVIEW_T:       return "REVIEW"
    return "REJECT"

print(f"Knowledge base ready: {len(SKILL_KB)} skills, {sum(len(v) for v in SKILL_KB.values())} aliases, "
      f"{len(DEGREE_KB)} degree rules, job = {JOB_TITLE}")

## Step 2: Input Dataset (50,000 resumes)
Real resumes are private personal data, so we **generate a realistic synthetic dataset** (this is allowed and clearly acknowledged in the report).
Every resume is plain text with **varied layouts** (different headings, section order, date formats, phone formats, skill-list styles) and **deliberate noise**:
synonyms/abbreviations (`ML`, `sklearn`), random upper/lower case, 2% spelling mistakes, company names that look like skills (`Excel Infotech`),
negations (`No hands-on experience with Docker`), "currently learning" statements, year-only dates, and unreadable degree lines.

Alongside each resume we store **hidden ground truth** (true skills, experience, degree, correct decision). The agent **never sees** these columns; they are used only to measure accuracy.

In [ ]:
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
FIRST = ["Aarav","Vivaan","Aditya","Sai","Arjun","Karthik","Rahul","Rohit","Suresh","Ramesh","Priya","Ananya","Sneha","Divya",
         "Lakshmi","Kavya","Pooja","Meghana","Harsha","Teja","Naveen","Varun","Manoj","Swathi","Bhavana","Deepak","Nikhil",
         "Sandeep","Anjali","Ishita","Mohit","Neha","Ravi","Sowmya","Vamsi","Charan","Tarun","Keerthi","Pranav","Rakesh"]
LAST = ["Reddy","Sharma","Kumar","Rao","Naidu","Patel","Singh","Gupta","Verma","Iyer","Nair","Chowdary","Varma","Raju","Das",
        "Mehta","Joshi","Pillai","Menon","Yadav","Shetty","Bhat","Kapoor","Jain","Mishra","Agarwal","Prasad","Murthy","Babu","Sastry"]
CITIES = ["Guntur","Hyderabad","Bengaluru","Chennai","Vijayawada","Visakhapatnam","Pune","Mumbai","Delhi","Kolkata",
          "Coimbatore","Noida","Ahmedabad","Kochi","Tirupati"]
DOMAINS = ["gmail.com","outlook.com","yahoo.com","mail.com"]
COMPANIES = ["Infosys","TCS","Wipro","HCL Technologies","Tech Mahindra","Cognizant","Capgemini","Accenture","Zoho","Freshworks",
             "Mindtree","Mphasis","Persistent Systems","Hexaware","Cyient","Sutherland","Excel Infotech","React Labs","Nextgen Solutions",
             "BrightPath Systems","Cloudnine Software","Vertex Analytics","Orion Digital","Pinnacle Tech","Quantum Leap Pvt Ltd",
             "BlueOcean Soft","Greenfield Systems","Skyline Infotech","Apex Software","Innova Labs"]
COLLEGES = ["JNTU Kakinada","Acharya Nagarjuna University","VIT Vellore","SRM University","KL University","Osmania University",
            "Anna University","Amrita Vishwa Vidyapeetham","NIT Warangal","IIIT Hyderabad","Manipal Institute of Technology",
            "PES University","Andhra University","BITS Pilani","Lovely Professional University"]
SOFT = ["Communication","Teamwork","Leadership","Problem Solving","Time Management","Critical Thinking","Tally",
        "Recruitment","Customer Handling","Presentation Skills"]
TASKS = ["Developed and maintained modules","Automated reporting workflows","Collaborated with cross-functional teams",
         "Built internal tools","Optimized application performance","Prepared dashboards for management",
         "Resolved production issues","Designed data pipelines","Wrote unit tests and documentation"]
PROJECTS = ["Student Management System","Sales Dashboard","Chat Application","Price Predictor","Image Classifier",
            "Library Portal","Attendance Tracker","Sentiment Analyzer","Expense Manager","Resume Parser"]

DEG_TABLE = {   # key: (surface forms, level, technical?)
    "btech_cs":    (["B.Tech in Computer Science","Bachelor of Technology (CSE)","B.Tech - Information Technology","B.E. in Computer Science and Engineering"], "Bachelor", True),
    "btech_other": (["B.Tech in Electronics and Communication","B.E. Mechanical Engineering","B.Tech in Civil Engineering"], "Bachelor", True),
    "bsc":   (["B.Sc. Computer Science","Bachelor of Science in Computer Science","BSc Mathematics"], "Bachelor", True),
    "bca":   (["BCA","Bachelor of Computer Applications"], "Bachelor", True),
    "mtech": (["M.Tech in Computer Science","Master of Technology (Computer Science)","M.Tech - Data Science"], "Master", True),
    "mca":   (["MCA","Master of Computer Applications"], "Master", True),
    "msc":   (["M.Sc. Data Science","Master of Science in Data Science"], "Master", True),
    "mba":   (["MBA","Master of Business Administration"], "Master", False),
    "bcom":  (["B.Com","Bachelor of Commerce"], "Bachelor", False),
    "ba":    (["B.A. English","Bachelor of Arts (Economics)"], "Bachelor", False),
    "bba":   (["BBA","Bachelor of Business Administration"], "Bachelor", False),
}
ARCH = {   # candidate archetypes: share, core skills, job titles
    "ml":      dict(p=0.22, core=["python","machine learning","pandas","numpy","scikit-learn","sql","deep learning","tensorflow","pytorch","nlp","statistics","computer vision"], titles=["Data Scientist","AI Engineer","Research Engineer"]),
    "analyst": dict(p=0.14, core=["sql","excel","power bi","tableau","python","data analysis","statistics","pandas","mysql"], titles=["Data Analyst","Business Analyst","BI Developer"]),
    "web":     dict(p=0.18, core=["javascript","html","css","react","node.js","mongodb","git","django","flask","python"], titles=["Web Developer","Frontend Developer","Full Stack Developer"]),
    "backend": dict(p=0.15, core=["java","sql","mysql","git","linux","docker","python","c++","aws"], titles=["Software Engineer","Backend Developer","Application Developer"]),
    "devops":  dict(p=0.10, core=["docker","kubernetes","aws","azure","linux","git","python"], titles=["DevOps Engineer","Cloud Engineer","Site Reliability Engineer"]),
    "nontech": dict(p=0.12, core=["excel"], titles=["Sales Executive","HR Executive","Accounts Assistant","Marketing Associate"]),
    "fresher": dict(p=0.09, core=["python","java","c++","sql","html","css","git","machine learning"], titles=["Trainee Engineer","Software Intern"]),
}
CERTABLE = ["aws","azure","python","tableau","power bi","docker","kubernetes","sql","machine learning","deep learning","tensorflow"]

def add_months(ym, k):
    y, m = ym; t = y * 12 + (m - 1) + k
    return (t // 12, t % 12 + 1)
def diff_months(a, b): return (b[0] - a[0]) * 12 + (b[1] - a[1])

def surface(skill, rng, noisy=True):
    # how a skill is written in a resume: alias choice, random casing, occasional typo
    al = SKILL_KB[skill]
    a = al[0] if (len(al) == 1 or rng.random() < 0.65) else rng.choice(al[1:])
    r = rng.random()
    if a in ("sql","aws","html","css","nlp","ml","js") and r < 0.8: a = a.upper()
    elif r < 0.5: a = a.title()
    if noisy and len(a) >= 6 and rng.random() < 0.02:       # 2% spelling mistakes
        i = rng.randrange(1, len(a) - 1); a = a[:i] + a[i + 1:]
    return a

def fmt_date(ym, style):
    y, m = ym
    return f"{MONTHS[m-1]} {y}" if style == "mon" else (f"{m:02d}/{y}" if style == "slash" else str(y))

def make_jobs(rng, arch):
    n = 0 if arch == "fresher" else rng.choices([0,1,2,3,4], [.2,.3,.3,.15,.05])[0]
    jobs = []
    if n == 0:
        if rng.random() < 0.4:                                   # a short internship
            end = add_months(NOW, -rng.randint(0, 8)); jobs.append((add_months(end, -rng.randint(2, 6)), end, False))
        return jobs
    present = rng.random() < 0.45
    end = NOW if present else add_months(NOW, -rng.randint(1, 24))
    for k in range(n):
        start = add_months(end, -rng.randint(6, 54))
        if start[0] < 2008: break
        jobs.append((start, end, present and k == 0))
        end = add_months(start, -rng.randint(0, 6))
    return jobs                                                  # latest job first

def render_resume(rng, i):
    arch = rng.choices(list(ARCH), weights=[ARCH[a]["p"] for a in ARCH])[0]
    first, last = rng.choice(FIRST), rng.choice(LAST); name = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}{rng.randint(1, 9999)}@{rng.choice(DOMAINS)}"
    p10 = rng.choice("6789") + "".join(rng.choice("0123456789") for _ in range(9))
    pf = rng.random()
    phone = p10 if pf < .3 else f"+91 {p10}" if pf < .6 else f"{p10[:5]} {p10[5:]}" if pf < .8 else f"+91-{p10[:5]}-{p10[5:]}"
    city = rng.choice(CITIES)

    # ---- ground-truth skills
    core = set(ARCH[arch]["core"]); skills = set()
    for s in SKILL_KB:
        p = (0.7 if arch == "nontech" else 0.6) if s in core else (0.015 if arch == "nontech" else 0.03)
        if rng.random() < p: skills.add(s)
    if arch == "nontech":
        if rng.random() < 0.08: skills.add("sql")
        if rng.random() < 0.10: skills.add("power bi")

    # ---- ground-truth degrees
    if arch == "nontech": top = rng.choices(["bcom","ba","mba","btech_other","bba"], [.3,.2,.2,.15,.15])[0]
    else: top = rng.choices(["btech_cs","btech_other","bsc","bca","mtech","mca","msc","mba","bcom"], [.30,.12,.10,.10,.12,.09,.08,.05,.04])[0]
    keys = [top]
    if top in ("mtech","mca","msc"): keys.append(rng.choice(["btech_cs","bsc","bca"]))
    if top == "mba": keys.append(rng.choice(["bcom","ba","btech_cs","btech_other"]))
    y = rng.randint(2010, 2026); edu_lines = []; deg_scores = []
    for j, k in enumerate(keys):
        forms, lvl, tech = DEG_TABLE[k]; deg_scores.append(DEGREE_SCORE[(lvl, tech)])
        span = 2 if lvl == "Master" else 4
        line = f"{rng.choice(forms)}, {rng.choice(COLLEGES)} ({y - span} - {y})"
        if rng.random() < .7: line += f", CGPA: {rng.uniform(6, 9.8):.1f}"
        edu_lines.append(line); y -= span
    if rng.random() < 0.02: edu_lines = [f"Engineering Graduate from {rng.choice(COLLEGES)}"]   # unreadable degree (noise)
    deg_true = max(deg_scores)

    # ---- ground-truth experience
    jobs = make_jobs(rng, arch); months = sum(diff_months(s, e) for s, e, _ in jobs); years = round(months / 12, 2)
    dstyle = rng.choices(["mon","slash","year"], [.5,.25,.25])[0]
    sep = rng.choice([" - ", " – ", " to "])
    listed = sorted(skills); rng.shuffle(listed)
    k_listed = int(np.ceil(0.75 * len(listed))) if listed else 0
    in_skills, hidden = listed[:k_listed], listed[k_listed:]

    # ---- sections
    secs = {}
    secs["summary"] = (rng.choice(HEADINGS["summary"]).upper(),
        f"{rng.choice(['Motivated','Dedicated','Detail-oriented','Enthusiastic'])} {rng.choice(ARCH[arch]['titles'])} "
        f"with a passion for building reliable solutions and learning new technologies.")
    items = [surface(s, rng) for s in in_skills] + rng.sample(SOFT, rng.randint(2, 4)); rng.shuffle(items)
    lay = rng.random()
    body = ", ".join(items) if lay < .4 else "\n".join("- " + t for t in items) if lay < .7 else " | ".join(items)
    cand = [s for s in SKILL_KB if s not in skills]; extra = []
    if rng.random() < 0.14:
        x = rng.choice(cand); extra.append(rng.choice([f"No hands-on experience with {surface(x, rng, False)}.",
            f"Not familiar with {surface(x, rng, False)}.", f"Currently learning {surface(x, rng, False)}.", f"Yet to learn {surface(x, rng, False)}."]))
    if rng.random() < 0.04:
        x, z = rng.sample(cand, 2); extra.append(f"No experience with {surface(x, rng, False)} and {surface(z, rng, False)}.")
    secs["skills"] = (rng.choice(HEADINGS["skills"]).upper(), body + ("\n" + "\n".join(extra) if extra else ""))
    if jobs:
        blocks = []
        for (s, e, pres) in jobs:
            end_txt = rng.choice(["Present","Current"]) if pres else fmt_date(e, dstyle)
            b = f"{rng.choice(ARCH[arch]['titles'])} | {rng.choice(COMPANIES)} | {fmt_date(s, dstyle)}{sep}{end_txt}"
            for _ in range(rng.randint(1, 2)):
                use = rng.sample(sorted(skills), min(len(skills), rng.randint(0, 2))) if skills else []
                b += "\n- " + rng.choice(TASKS) + (" using " + " and ".join(surface(u, rng) for u in use) if use else "") + "."
            blocks.append(b)
        exp_body = "\n".join(blocks)
    else:
        exp_body = "Fresher - seeking first full-time opportunity."
    secs["exp"] = (rng.choice(HEADINGS["exp"]).title() + (":" if rng.random() < .3 else ""), exp_body)
    secs["edu"] = (rng.choice(HEADINGS["edu"]).upper(), "\n".join(edu_lines))
    if hidden or rng.random() < .85:
        pl, pool, hid = [], sorted(skills), list(hidden)
        for _ in range(max(1, int(np.ceil(len(hid) / 3)))):
            take = hid[:3]; hid = hid[3:]
            if not take and pool: take = rng.sample(pool, min(len(pool), rng.randint(1, 2)))
            pl.append(f"{rng.choice(PROJECTS)}: Built a working prototype" + (" using " + ", ".join(surface(t, rng) for t in take) if take else "") + ".")
        secs["proj"] = (rng.choice(HEADINGS["proj"]).upper(), "\n".join(pl))
    certs = [s for s in sorted(skills) if s in CERTABLE]
    if certs and rng.random() < .35:
        secs["cert"] = ("CERTIFICATIONS", "\n".join(f"{surface(s, rng)} Certification - Coursera" for s in rng.sample(certs, min(2, len(certs)))))
    order = list(secs); rng.shuffle(order)
    if rng.random() < .7: order.remove("summary"); order.insert(0, "summary")

    hs = rng.random()
    if hs < .5: header = [name.upper(), f"Email: {email} | Phone: {phone} | {city}"]
    elif hs < .8: header = [name, city, email, phone]
    else: header = ["CURRICULUM VITAE", f"Name: {name}", f"Email: {email}", f"Mobile: {phone}", f"Location: {city}"]
    text = "\n".join(header) + "\n\n" + "\n\n".join(f"{secs[k][0]}\n{secs[k][1]}" for k in order)

    u, req_n, _ = utility(skills, years, deg_true)
    return dict(resume_id=f"R{i:05d}", resume_text=text, archetype=arch, true_name=name, true_email=email, true_phone=p10,
                true_skills="|".join(sorted(skills)), true_years=years, true_degree_score=deg_true,
                true_utility=u, true_label=decide(u, req_n))

def generate_resumes(n, seed=SEED):
    rng = random.Random(seed)
    return pd.DataFrame([render_resume(rng, i) for i in range(1, n + 1)])

t0 = time.time()
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH); print(f"Loaded existing dataset: {DATA_PATH}")
else:
    df = generate_resumes(N_RESUMES); df.to_csv(DATA_PATH, index=False); print(f"Generated and saved: {DATA_PATH}")
df["true_skills"] = df["true_skills"].fillna("")
df["true_phone"] = df["true_phone"].astype(str)
print(f"{len(df):,} resumes | {time.time()-t0:.1f} s | file size ~ {os.path.getsize(DATA_PATH)/1e6:.1f} MB")

In [ ]:
print("Columns:", list(df.columns))
print("\nGround-truth decision distribution:")
print(df["true_label"].value_counts().to_string())
print("\nAverage resume length: %.0f characters" % df["resume_text"].str.len().mean())
print("\n" + "=" * 70 + "\nSAMPLE INPUT RESUME (what the agent receives)\n" + "=" * 70)
print(df.loc[7, "resume_text"])

## Step 3: The Intelligent Agent

**PEAS description**

| PEAS | Our agent |
|---|---|
| **P**erformance measure | Skill-extraction F1, experience error, accuracy of SHORTLIST / REVIEW / REJECT decisions, speed |
| **E**nvironment | A stream of unstructured resume documents + one job profile (partially observable, text is noisy, static per resume, single agent) |
| **A**ctuators | Structured record (name, email, phone, skills, years, degree), utility score, decision and reasons |
| **S**ensors | Raw resume text |

**Agent type: utility-based, knowledge-based agent.**
`Perceive` (split document into sections) -> `Reason` (use the knowledge base to extract facts; apply rules such as negation handling) -> `Evaluate` (utility function scores the candidate against the job) -> `Act` (SHORTLIST / REVIEW / REJECT + explanation).

In [ ]:
TOK   = r"(?:[A-Za-z]{3,9}\.?\s+\d{4}(?!\d)|(?<!\d)\d{1,2}/\d{4}(?!\d)|(?<![\d/])\d{4}(?![\d/])" + r")"
RANGE = re.compile(rf"({TOK})\s*(?:-|–|—|\bto\b)\s*({TOK}|Present|Current|Till date)", re.I)
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]*\w")
PHONE = re.compile(r"(?:\+91[\s-]?)?[6-9]\d{4}[\s-]?\d{5}")
NEG   = re.compile(r"(no (hands-on )?(experience|exposure|knowledge)|not (familiar|experienced)|never (used|worked)|"
                   r"currently learning|yet to learn|plan(ning)? to learn)")

def parse_tok(t):
    t = t.strip()
    if t.lower() in ("present", "current", "till date"): return NOW
    m = re.match(r"([A-Za-z]{3,9})\.?\s+(\d{4})$", t)
    if m and m.group(1)[:3].title() in MONTHS: return (int(m.group(2)), MONTHS.index(m.group(1)[:3].title()) + 1)
    m = re.match(r"(\d{1,2})/(\d{4})$", t)
    if m: return (int(m.group(2)), int(m.group(1)))
    m = re.match(r"(\d{4})$", t)
    return (int(m.group(1)), 1) if m else None

class ResumeScreeningAgent:
    # Utility-based, knowledge-based agent for resume screening.
    # Flags exist only so the ablation study can switch individual knowledge components off.
    def __init__(self, use_alias=True, use_negation=True, use_sections=True):
        self.use_negation, self.use_sections = use_negation, use_sections
        self.alias_map = {a.lower(): s for s, als in SKILL_KB.items() for a in (als if use_alias else als[:1])}
        pat = "|".join(re.escape(a) for a in sorted(self.alias_map, key=len, reverse=True))
        self.skill_re = re.compile(r"(?<![A-Za-z0-9+#])(?:" + pat + r")(?![A-Za-z0-9+#])", re.I)

    # ---------- PERCEIVE ----------
    def split_sections(self, text):
        secs, cur = {"header": []}, "header"
        for line in text.split("\n"):
            key = line.strip().rstrip(":").lower()
            if key in HEADING_MAP: cur = HEADING_MAP[key]; secs.setdefault(cur, []); continue
            secs.setdefault(cur, []).append(line)
        return {k: "\n".join(v) for k, v in secs.items()}

    # ---------- REASON (knowledge-based extraction) ----------
    def _negated(self, text, start):
        w = text[max(0, start - 70):start]
        cut = max(w.rfind("\n"), w.rfind(";"), w.rfind(". "))
        if cut >= 0: w = w[cut + 1:]
        return bool(NEG.search(w.lower()))

    def extract_skills(self, text):
        found = set()
        for m in self.skill_re.finditer(text):
            if self.use_negation and self._negated(text, m.start()): continue
            found.add(self.alias_map[m.group(0).lower()])
        return found

    def experience_years(self, text, secs):
        txt = secs.get("exp", "") if self.use_sections else text
        months = 0
        for m in RANGE.finditer(txt):
            a, b = parse_tok(m.group(1)), parse_tok(m.group(2))
            if a and b and diff_months(a, b) > 0: months += diff_months(a, b)
        return round(months / 12, 2)

    def degree(self, text, secs):
        txt = secs.get("edu") if (self.use_sections and secs.get("edu")) else text
        best = (0.0, "Not found")
        for rx, lvl, tech, lab in DEGREE_KB:
            if rx.search(txt) and DEGREE_SCORE[(lvl, tech)] > best[0]: best = (DEGREE_SCORE[(lvl, tech)], f"{lvl} ({lab})")
        return best

    def contact(self, text, secs):
        head = secs.get("header", "")
        m = re.search(r"^name:\s*(.+)$", head, re.I | re.M)
        if m: name = m.group(1).strip()
        else:
            lines = [l.strip() for l in head.split("\n") if l.strip() and l.strip().lower() not in ("curriculum vitae", "resume")]
            name = lines[0] if lines else ""
        em, ph = EMAIL.search(head), PHONE.search(head)
        return name.title(), (em.group(0) if em else ""), (re.sub(r"\D", "", ph.group(0))[-10:] if ph else "")

    # ---------- EVALUATE + ACT ----------
    def run(self, text):
        secs = self.split_sections(text)
        name, email, phone = self.contact(text, secs)
        skills = self.extract_skills(text)
        years = self.experience_years(text, secs)
        d_score, d_label = self.degree(text, secs)
        u, req_n, pref_n = utility(skills, years, d_score)
        decision = decide(u, req_n)
        missing = [s for s in REQUIRED if s not in skills]
        reasons = [f"{req_n}/{len(REQUIRED)} required skills" + (f" (missing: {', '.join(missing)})" if missing else ""),
                   f"{pref_n} preferred skills", f"{years} yrs experience (target {MIN_EXP_YEARS:g})", f"degree: {d_label}"]
        return dict(name=name, email=email, phone=phone, skills=skills, years=years, degree=d_label, degree_score=d_score,
                    utility=u, decision=decision, reasons="; ".join(reasons))

agent = ResumeScreeningAgent()
print("Agent created. Test on the sample resume:\n")
out = agent.run(df.loc[7, "resume_text"])
for k, v in out.items(): print(f"{k:13s}: {sorted(v) if isinstance(v, set) else v}")

## Step 4: Run the Agent on All 50,000 Resumes

In [ ]:
t0 = time.time()
results = [agent.run(t) for t in df["resume_text"]]
agent_time = time.time() - t0
res = pd.DataFrame(results)
print(f"Screened {len(res):,} resumes in {agent_time:.1f} s  ({len(res)/agent_time:,.0f} resumes/second, "
      f"{1000*agent_time/len(res):.2f} ms per resume)")
print("\nAgent decisions:")
print(res["decision"].value_counts().to_string())
res.head(5)[["name","email","phone","years","degree","utility","decision"]]

## Step 5: Evaluation
Because we know the hidden ground truth, we can measure accuracy for each sub-task and for the final decision.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def to_set(s): return set(s.split("|")) if s else set()
truth_sets = [to_set(s) for s in df["true_skills"]]

def skill_prf(truth, pred):
    tp = sum(len(t & p) for t, p in zip(truth, pred)); fp = sum(len(p - t) for t, p in zip(truth, pred)); fn = sum(len(t - p) for t, p in zip(truth, pred))
    P = tp / (tp + fp); R = tp / (tp + fn); return P, R, 2 * P * R / (P + R)

P, R, F = skill_prf(truth_sets, list(res["skills"]))
mae = (res["years"] - df["true_years"]).abs().mean()
within = ((res["years"] - df["true_years"]).abs() <= 0.5).mean()
deg_acc = (res["degree_score"] == df["true_degree_score"]).mean()
name_acc = (res["name"] == df["true_name"]).mean(); mail_acc = (res["email"] == df["true_email"]).mean(); ph_acc = (res["phone"] == df["true_phone"]).mean()
acc = accuracy_score(df["true_label"], res["decision"]); mf1 = f1_score(df["true_label"], res["decision"], average="macro")

print("INFORMATION EXTRACTION")
print(f"  Skills      precision {P:.4f} | recall {R:.4f} | F1 {F:.4f}")
print(f"  Experience  MAE {mae:.3f} years | within +/-0.5 yr: {within:.2%}")
print(f"  Degree      accuracy {deg_acc:.2%}")
print(f"  Name {name_acc:.2%} | Email {mail_acc:.2%} | Phone {ph_acc:.2%}")
print("\nSCREENING DECISION (SHORTLIST / REVIEW / REJECT)")
print(f"  Accuracy {acc:.2%} | Macro-F1 {mf1:.4f}")
print(classification_report(df["true_label"], res["decision"], digits=3))

In [ ]:
labels = ["SHORTLIST", "REVIEW", "REJECT"]
cm = confusion_matrix(df["true_label"], res["decision"], labels=labels)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
im = ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(labels); ax[0].set_yticks(range(3)); ax[0].set_yticklabels(labels)
ax[0].set_xlabel("Agent decision"); ax[0].set_ylabel("True decision"); ax[0].set_title("Confusion matrix (50,000 resumes)")
for i in range(3):
    for j in range(3): ax[0].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
x = np.arange(3); w = 0.38
ax[1].bar(x - w/2, [(df["true_label"] == l).sum() for l in labels], w, label="Ground truth")
ax[1].bar(x + w/2, [(res["decision"] == l).sum() for l in labels], w, label="Agent")
ax[1].set_xticks(x); ax[1].set_xticklabels(labels); ax[1].set_title("Decision distribution"); ax[1].legend()
plt.tight_layout(); plt.show()

## Step 6: Comparison with Alternative Approaches + Ablation Study
**Baseline 1: Keyword matching.** Looks for the plain skill names as substrings. No synonyms, no negation handling, no experience/degree parsing.
**Baseline 2: TF-IDF similarity.** Converts the resume and the job description into TF-IDF vectors and uses cosine similarity. We *give it an advantage* by choosing its cut-offs so that it predicts exactly the true share of each class.

In [ ]:
def keyword_baseline(text):
    t = text.lower(); skills = {s for s in SKILL_KB if s in t}
    req_n = sum(s in skills for s in REQUIRED); pref_n = sum(s in skills for s in PREFERRED)
    u = (WEIGHTS["req"] * req_n / len(REQUIRED) + WEIGHTS["pref"] * min(1, pref_n / 5)) / (WEIGHTS["req"] + WEIGHTS["pref"])
    return decide(u, req_n), skills

t0 = time.time(); kb = [keyword_baseline(t) for t in df["resume_text"]]; kw_time = time.time() - t0
kw_pred = [k[0] for k in kb]; kw_sk = [k[1] for k in kb]

from sklearn.feature_extraction.text import TfidfVectorizer
JD_TEXT = ("python machine learning sql deep learning nlp tensorflow pytorch scikit-learn pandas aws docker git "
           "bachelor master computer science degree 3 years experience")
t0 = time.time()
vec = TfidfVectorizer(sublinear_tf=True, min_df=5, ngram_range=(1, 2), stop_words="english")
X = vec.fit_transform(df["resume_text"]); sim = (X @ vec.transform([JD_TEXT]).T).toarray().ravel()
share_s, share_r = (df["true_label"] == "SHORTLIST").mean(), (df["true_label"] == "REVIEW").mean()
c1, c2 = np.quantile(sim, 1 - share_s), np.quantile(sim, 1 - share_s - share_r)
tf_pred = np.where(sim >= c1, "SHORTLIST", np.where(sim >= c2, "REVIEW", "REJECT")); tf_time = time.time() - t0

def row(name, pred, sk, secs):
    r = dict(Method=name, Accuracy=accuracy_score(df["true_label"], pred), MacroF1=f1_score(df["true_label"], pred, average="macro"),
             Shortlist_F1=f1_score(df["true_label"], pred, labels=["SHORTLIST"], average="macro"), Seconds=secs)
    r["Skill_F1"] = skill_prf(truth_sets, sk)[2] if sk is not None else np.nan
    return r
cmp_df = pd.DataFrame([row("Utility-based KB Agent (ours)", res["decision"], list(res["skills"]), agent_time),
                       row("Keyword matching", kw_pred, kw_sk, kw_time),
                       row("TF-IDF cosine similarity", tf_pred, None, tf_time)]).round(4)
display(cmp_df) if "display" in globals() else print(cmp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(cmp_df["Method"], cmp_df["Accuracy"] * 100, color=["#2E75B6", "#999999", "#BBBBBB"])
for i, v in enumerate(cmp_df["Accuracy"] * 100): ax.text(i, v + 1, f"{v:.1f}%", ha="center")
ax.set_ylim(0, 105); ax.set_ylabel("Decision accuracy (%)"); ax.set_title("Agent vs alternative approaches")
plt.xticks(rotation=12); plt.tight_layout(); plt.show()

In [ ]:
# Ablation: switch off one piece of knowledge at a time (10,000-resume sample to keep it quick)
sub = df.sample(10_000, random_state=SEED).reset_index(drop=True); sub_truth = [to_set(s) for s in sub["true_skills"]]
configs = {"Full agent": {}, "No synonym knowledge (aliases off)": dict(use_alias=False),
           "No negation handling": dict(use_negation=False), "No section understanding": dict(use_sections=False)}
rows = []
for name, kw in configs.items():
    a = ResumeScreeningAgent(**kw); t0 = time.time(); out = [a.run(t) for t in sub["resume_text"]]; sec = time.time() - t0
    o = pd.DataFrame(out)
    rows.append(dict(Configuration=name, Skill_F1=skill_prf(sub_truth, list(o["skills"]))[2],
                     Exp_MAE_years=(o["years"] - sub["true_years"]).abs().mean(),
                     Decision_Accuracy=accuracy_score(sub["true_label"], o["decision"]), Seconds=sec))
abl = pd.DataFrame(rows).round(4)
display(abl) if "display" in globals() else print(abl.to_string(index=False))

## Step 7: Error Analysis, Ranked Shortlist and Live Demo

In [ ]:
wrong = df.index[df["true_label"] != res["decision"]]
causes = Counter()
for i in wrong:
    t, p = truth_sets[i], res.at[i, "skills"]
    if t - p: causes["missed a real skill (typo / unseen wording)"] += 1
    if p - t: causes["counted a skill that is not real (e.g. company name 'Excel Infotech')"] += 1
    if abs(res.at[i, "years"] - df.at[i, "true_years"]) > 0.5: causes["experience off by more than 0.5 yr (year-only dates)"] += 1
    if res.at[i, "degree_score"] != df.at[i, "true_degree_score"]: causes["degree not recognised"] += 1
print(f"Wrong decisions: {len(wrong):,} of {len(df):,} ({len(wrong)/len(df):.2%})")
print("Contributing factors (one wrong decision can have several):")
for c, n in causes.most_common(): print(f"  {n:5d}  {c}")

i = wrong[0] if len(wrong) else None
if i is not None:
    print(f"\nExample error -> {df.at[i, 'resume_id']}: true = {df.at[i, 'true_label']}, agent = {res.at[i, 'decision']}")
    print("  true skills :", sorted(truth_sets[i])); print("  found skills:", sorted(res.at[i, "skills"]))
    print(f"  true years {df.at[i, 'true_years']} vs agent {res.at[i, 'years']}")

In [ ]:
# Ranked shortlist: the output a recruiter actually uses
top = res.assign(resume_id=df["resume_id"])
top = top[top["decision"] == "SHORTLIST"].assign(skills_found=lambda d: d["skills"].apply(len)).sort_values(["utility", "skills_found", "years"], ascending=False).head(10)
print(f"TOP 10 CANDIDATES for: {JOB_TITLE}\n")
print(top[["resume_id", "name", "email", "years", "degree", "skills_found", "utility"]].to_string(index=False))

out_df = res.copy(); out_df.insert(0, "resume_id", df["resume_id"]); out_df["skills"] = out_df["skills"].apply(lambda s: "|".join(sorted(s)))
out_df.to_csv("screening_results.csv", index=False); print("\nSaved: screening_results.csv (all 50,000 structured records + decisions)")
try:
    from google.colab import files; files.download("screening_results.csv")
except Exception: pass

In [ ]:
# LIVE DEMO: paste any resume text between the triple quotes and run this cell
demo_resume = """
PRIYA SHARMA
Email: priya.sharma@gmail.com | Phone: +91 98765 43210 | Hyderabad

SUMMARY
Enthusiastic Data Scientist with a passion for building reliable solutions.

TECHNICAL SKILLS
Python, ML, Deep Learning, SQL, sklearn, Pandas, AWS
No hands-on experience with Kubernetes.

WORK EXPERIENCE
Data Scientist | Zoho | Jun 2021 - Present
- Built churn prediction models using TensorFlow and Docker.
AI Engineer | Cognizant | Jan 2019 - May 2021

EDUCATION
M.Tech in Computer Science, JNTU Kakinada (2017 - 2019)
"""
r = ResumeScreeningAgent().run(demo_resume)
print(f"Candidate : {r['name']} | {r['email']} | {r['phone']}")
print(f"Skills    : {sorted(r['skills'])}")
print(f"Experience: {r['years']} years | Degree: {r['degree']}")
print(f"Utility   : {r['utility']}  ->  DECISION: {r['decision']}")
print(f"Reasons   : {r['reasons']}")

## Summary
* **Input:** 50,000 unstructured resume texts + one job profile (required / preferred skills, minimum experience, degree preference).
* **Output:** structured candidate record (name, email, phone, skills, experience, degree), utility score, SHORTLIST / REVIEW / REJECT decision with reasons, and a ranked shortlist.
* **Method:** utility-based, knowledge-based intelligent agent. State-space search (BFS / A*) and adversarial search do not apply, because resume screening has no path to find and no opponent; a black-box ML/LLM model would need labelled data and cannot explain its decisions.
* **Evidence:** see the evaluation, comparison and ablation tables above.

*Dataset note:* the resumes are synthetically generated for privacy; the same agent can be applied unchanged to real resume text.